In [0]:
# %pip install -r https://raw.githubusercontent.com/seaninc-training/databricks-ai-project/refs/heads/main/requirements.txt

In [0]:
# Imports and Variable Set Up
import os
import logging
import uuid
import time
from datetime import datetime
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from databricks.vector_search.client import VectorSearchClient
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Load foundations config
foundations = {
    row['config_key']: row['config_value']
    for row in spark.sql("SELECT config_key, config_value FROM workspace.ai_project.foundations").collect()
}

catalog        = foundations['catalog']
schema         = foundations['schema']
volume         = foundations['volume']
endpoint_name  = foundations['endpoint_name']
embedding_model = foundations['embedding_model']

# Base Volume Path
vol_path = f"/Volumes/{catalog}/{schema}/{volume}/"

# Load source_config from Delta table
source_config_rows = spark.sql("SELECT * FROM workspace.ai_project.source_config").collect()
source_config = {
    row['source_type']: {
        "landing_path":   row['landing_path'],
        "processed_path": row['processed_path'],
        "table":          row['chunks_table'],
        "min_size_kb":    row['min_size_kb'],
        "index":          row['index_name']
    }
    for row in source_config_rows
}

# Vector Search client
vsc = VectorSearchClient()

# File config
valid_extensions = ('.txt', '.pdf', '.md')

In [0]:
%run ../utils/logging_utils

In [0]:
%run ../utils/search_utils

In [0]:
all_files_to_process = {}

def get_files_to_process(landing_path, source_type):

    if source_type is None:
        raise ValueError("source_type must be specified: 'books' or 'docs'")
    elif source_type not in source_config:
        raise ValueError(f"Invalid source_type '{source_type}'. Must be one of: {list(source_config.keys())}")
    
    to_process = []
    min_size_kb = source_config[source_type]["min_size_kb"]
    
    try:
        raw_files = dbutils.fs.ls(landing_path)
        if not raw_files:
            logger.warning(f"⚠️ Source folder is empty: {landing_path}")
        else:
            logger.info(f"✅ Found {len(raw_files)} files to process.")
            for file in raw_files:
                if file.name.lower().endswith(valid_extensions):
                    size_kb = file.size / 1024
                    if size_kb < min_size_kb:
                        logger.warning(f"⚠️ Skipping {file.name}: File is too small ({size_kb:.2f} KB).")
                        if source_type == "docs":
                            deactivate_doc_source(file.name, reason="too_small")
                        dbutils.fs.rm(file.path)
                        continue
                    to_process.append({
                        "path": file.path,
                        "name": file.name,
                        "type": "pdf" if file.name.lower().endswith(".pdf") else "text",
                        "source_type": source_type
                    })
                    logger.info(f"📖 {file.name} validated ({size_kb:.2f} KB).")
                else:
                    logger.warning(f"⚠️ {file} is not a permitted file type.")
                    dbutils.fs.rm(file.path)
                    continue

            if len(to_process) == 0:
                logger.warning(f"⚠️ No valid files found in {landing_path}.")
            else:
                logger.info(f"✅ Total files ready for ingestion: {len(to_process)}")

    except Exception as e:
        logger.error(f"❌ Error accessing volume: {e}")
    
    return to_process


for source_type, config in source_config.items():
    all_files_to_process[source_type] = get_files_to_process(config["landing_path"], source_type)

In [0]:
# Set up text splitter for chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    add_start_index=True,
    separators=["\n\n", "\n", ". ", " ", ""])


def process_files(files_to_process, source_type):
    config = source_config[source_type]
    chunks_table = config["table"]
    processed_path = config["processed_path"]

    for file_info in files_to_process:
        try:
            logger.info(f"🚀 Processing: {file_info['name']}")

            local_path = file_info['path'].replace("dbfs:", "")
            
            if file_info['type'] == "pdf":
                loader = PyPDFLoader(local_path)
            else:
                loader = TextLoader(local_path, encoding="utf-8")

            raw_docs = loader.load()

            # Create Chunks
            chunks = text_splitter.split_documents(raw_docs)

            # Define explicit schema to prevent type inference issues
            chunks_schema = StructType([
                StructField("chunk_id", StringType(), False),
                StructField("content", StringType(), True),
                StructField("source", StringType(), True),
                StructField("type", StringType(), True),
                StructField("page_number", IntegerType(), True),
                StructField("start_index", IntegerType(), True),
                StructField("created_at", TimestampType(), True)
            ])
            
            # Prepare data for Vector Search
            data = [{
                "chunk_id": str(uuid.uuid4()),
                "content": chunk.page_content,
                "source": file_info['name'],
                "type": file_info['type'],
                "page_number": int(chunk.metadata.get("page", 1)),
                "start_index": int(chunk.metadata.get("start_index", 0)),
                "created_at": datetime.now()
            } for chunk in chunks]

            # Convert to Spark DF with explicit schema
            df = spark.createDataFrame(data, schema=chunks_schema)

            # Write to Delta with CDF Enabled
            if not spark.catalog.tableExists(chunks_table):
                (df.write.format("delta")
                   .option("delta.enableChangeDataFeed", "true")
                   .mode("overwrite")
                   .saveAsTable(chunks_table))
                logger.info(f"✨ Created new table: {chunks_table}")
            else:
                df.write.format("delta").mode("append").saveAsTable(chunks_table)
                logger.info(f"➕ Appended {len(chunks)} chunks to {chunks_table}")

            # Move file to processed folder
            destination = f"{processed_path}/{file_info['name']}"
            dbutils.fs.mv(local_path, destination)
            logger.info(f"✅ Processed and moved: {file_info['name']}")

            # Log success
            write_processing_log(
                file_name=file_info['name'],
                source_type=source_type,
                target_table=chunks_table,
                status="SUCCESS",
                chunk_count=len(chunks)
            )

        except Exception as e:
            logger.error(f"❌ Failed to process {file_info['name']}: {e}")
            # Log failure

            write_processing_log(
                file_name=file_info['name'],
                source_type=source_type,
                target_table=chunks_table,
                status="FAILED",
                error_message=str(e)
            )

    logger.info(f"🏁 All files processed for table: {chunks_table}")


for source_type, files in all_files_to_process.items():
    if files:
        process_files(files, source_type)
    else:
        logger.info(f"ℹ️ No files to process for source type: {source_type}")

In [0]:
# Idempotent Vector Search Setup

# 1. Ensure Endpoint exists
if not any(e['name'] == endpoint_name for e in vsc.list_endpoints().get('endpoints', [])):
    logger.info(f"🚀 Creating endpoint '{endpoint_name}'...")
    try:
        vsc.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")
        vsc.wait_for_endpoint(endpoint_name)
        logger.info(f"🟢 Endpoint '{endpoint_name}' is now ONLINE.")
    except Exception as e:
        logger.error(f"❌ Failed to create endpoint: {e}")
else:
    logger.info(f"✅ Endpoint '{endpoint_name}' is ready.")

# 2. Ensure Indexes exist and sync if needed
for source_type, config in source_config.items():
    index = config["index"]
    chunks_table = config["table"]
    files_processed = len(all_files_to_process.get(source_type, []))

    if not vsc.index_exists(endpoint_name=endpoint_name, index_name=index):
        logger.info(f"✨ Creating new index '{index}'...")
        try:
            vsc.create_delta_sync_index(
                endpoint_name=endpoint_name,
                source_table_name=chunks_table,
                index_name=index,
                pipeline_type="TRIGGERED",
                primary_key="chunk_id",
                embedding_source_column="content",
                embedding_model_endpoint_name=embedding_model
            )

            # Wait for this index before creating the next one
            logger.info(f"⏳ Waiting for '{index}' to come ONLINE before proceeding...")
            vsc.get_index(endpoint_name, index).wait_until_ready()
            logger.info(f"🟢 Index '{index}' is ONLINE.")

        except Exception as e:
            logger.error(f"❌ Failed to create index '{index}': {e}")
    else:
        logger.info(f"✅ Index '{index}' already exists.")

        # Smart Sync: Only sync if new files were processed
        if files_processed > 0:
            try:
                logger.info(f"🔄 New data detected ({files_processed} files). Triggering sync for '{index}'...")
                vsc.get_index(endpoint_name, index).sync()

                # Wait for this index before creating the next one
                logger.info(f"⏳ Waiting for '{index}' to come ONLINE before proceeding...")
                vsc.get_index(endpoint_name, index).wait_until_ready()
                logger.info(f"🟢 Index '{index}' is ONLINE.")
                
            except Exception as e:
                logger.error(f"❌ Failed to sync index '{index}': {e}")
        else:
            logger.info(f"ℹ️ No new files for '{source_type}'. Skipping sync.")